In [1]:
# ==========================================
# AI MEETING SUMMARIZER - SETUP
# ==========================================

!apt-get update -qq
!apt-get install -y zstd ffmpeg -qq

!curl -fsSL https://ollama.com/install.sh | sh

!pip install -q gradio pypdf openai-whisper requests

import subprocess
import time
import requests
import whisper

# Start Ollama server in background
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait for Ollama
print("Starting Ollama...")
for i in range(30):
    try:
        requests.get("http://localhost:11434", timeout=2)
        print("✅ Ollama server is running")
        break
    except:
        time.sleep(1)
else:
    raise RuntimeError("❌ Ollama failed to start")

# Download model
print("Downloading Llama 3.2...")
subprocess.run(["ollama", "pull", "llama3.2"], check=True)

print("✅ Llama 3.2 ready")

# Load Whisper
print("Loading Whisper model...")
whisper_model = whisper.load_model("base")

print("✅ Whisper loaded")
print("✅ Setup complete")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 9.3 MB/s eta 0:00:0

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 155MiB/s]


✅ Whisper loaded
✅ Setup complete


In [2]:
# ==========================================
# AI MEETING SUMMARIZER
# ==========================================

import os
import requests
import gradio as gr
from pypdf import PdfReader


# ------------------------------------------
# OLLAMA
# ------------------------------------------

def ask_ollama(prompt):

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "llama3.2",
                "prompt": prompt,
                "stream": False
            },
            timeout=300
        )

        response.raise_for_status()

        return response.json()["response"]

    except Exception as e:
        return f"❌ Ollama Error: {e}"


# ------------------------------------------
# MEETING SUMMARIZATION
# ------------------------------------------

def summarize_meeting(transcript):

    prompt = f"""
You are a professional AI meeting assistant.

Analyze the meeting transcript and create a clear,
professional meeting report.

Use exactly these sections:

# MEETING SUMMARY
Give a concise summary of the meeting.

# KEY DISCUSSION POINTS
List the important topics discussed.

# DECISIONS MADE
List only decisions explicitly mentioned.
Do not invent decisions.

# ACTION ITEMS
For every task mentioned, provide:
- Task
- Responsible person
- Deadline

If information is unavailable, write "Not specified".

# DEADLINES
List all deadlines mentioned.

# UNRESOLVED ISSUES
List questions, problems, or decisions that remain unresolved.

IMPORTANT:
- Do not invent information.
- Use only information present in the transcript.
- Keep the report clear and professional.

MEETING TRANSCRIPT
==================
{transcript}
==================
"""

    return ask_ollama(prompt)


# ------------------------------------------
# AUDIO TRANSCRIPTION
# ------------------------------------------

def transcribe_audio(audio_file):

    try:
        result = whisper_model.transcribe(audio_file)
        return result["text"]

    except Exception as e:
        return f"❌ Transcription Error: {e}"


# ------------------------------------------
# FILE TEXT EXTRACTION
# ------------------------------------------

def extract_text(file_path):

    extension = os.path.splitext(file_path)[1].lower()

    if extension == ".txt":

        with open(file_path, "r", encoding="utf-8") as file:
            return file.read()

    elif extension == ".pdf":

        reader = PdfReader(file_path)

        return "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    return ""


# ------------------------------------------
# MAIN PROCESS
# ------------------------------------------

def process_meeting(pasted_text, document_file, audio_file):

    if pasted_text and pasted_text.strip():

        transcript = pasted_text.strip()

    elif document_file:

        transcript = extract_text(document_file)

    elif audio_file:

        transcript = transcribe_audio(audio_file)

    else:

        return "❌ Please provide a transcript, document, or audio file."

    if not transcript.strip():

        return "❌ No text could be extracted."

    return summarize_meeting(transcript)


# ------------------------------------------
# GRADIO INTERFACE
# ------------------------------------------

with gr.Blocks(title="AI Meeting Summarizer") as demo:

    gr.Markdown("""
    # 🤖 AI Meeting Summarizer

    **Transform meeting transcripts, documents, and recordings
    into structured meeting reports using AI.**
    """)

    with gr.Row():

        # INPUT SECTION
        with gr.Column():

            gr.Markdown("### 📥 Meeting Input")

            text_input = gr.Textbox(
                label="Meeting Transcript",
                placeholder="Paste your meeting transcript here...",
                lines=12
            )

            document_input = gr.File(
                label="Upload TXT / PDF",
                file_types=[".txt", ".pdf"],
                type="filepath"
            )

            audio_input = gr.Audio(
                label="Upload Meeting Recording",
                sources=["upload"],
                type="filepath"
            )

            summarize_button = gr.Button(
                "🚀 Generate Meeting Report",
                variant="primary"
            )

        # OUTPUT SECTION
        with gr.Column():

            gr.Markdown("### 📊 Meeting Report")

            output = gr.Markdown(
                value="Your meeting report will appear here."
            )

    summarize_button.click(
        fn=process_meeting,
        inputs=[
            text_input,
            document_input,
            audio_input
        ],
        outputs=output
    )


# ------------------------------------------
# LAUNCH
# ------------------------------------------

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b120f74da7524de146.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
demo.launch(share=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b120f74da7524de146.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
